In [ ]:
import numpy as np
import pandas as pd
import sklearn as sk
from utils.flux_integration import Integrating_Flux
from utils.phase_detection import Find_Proton_Flux_Phases
import matplotlib.pyplot as plt

# Ex 1.  GOES-16 SEU timestamped data with SOHO Proton Flux

## Linear Regression of SEU instances and Proton Fluxes



In [ ]:
goes_16_seu_df = pd.read_excel("data/seu_timestamp/g16_g17_exs_spw.xlsx", sheet_name= "g16_exs_spw" )

goes_16_proton_flux = pd.read_csv("data/enviromental/ephin-soho/proton_flux_2016Dec-2022Mar_EPHINSOHO.lst", header = None, names = ['Year', 'DOY', 'Hour', 'P4', 'P8', 'P25', 'P41'], sep = "\s+")
goes_16_proton_flux = Integrating_Flux(goes_16_proton_flux)

goes_16_proton_flux_phases = Find_Proton_Flux_Phases(goes_16_proton_flux, 200, "goes16_proton_flux_phases")
goes_16_proton_flux

In [ ]:
goes_16_seu_df["Count"] = np.ones(goes_16_seu_df.shape[0])

goes_16_seu_rate_df = goes_16_seu_df.groupby(pd.Grouper(key= "Datetime", freq = "D")).sum() 
goes_16_proton_flux_mean = goes_16_proton_flux.groupby(pd.Grouper(key = "Timestamp", freq = "D")).mean()



proton_flux_seu_rate_df = pd.concat([goes_16_seu_rate_df["Count"], goes_16_proton_flux_mean["P_tot"]], axis = 1, keys = ['SEU/h', "Flux/h"]).dropna()


fig, ax1 = plt.subplots()

# -----------------------------------
# LEFT AXIS (FRONT)
# -----------------------------------

ax1.set_zorder(2)
ax1.patch.set_visible(False)

ax1.set_title("SEU Rate vs Proton Flux")

ax1.set_xlabel("Time Unit")

ax1.set_ylabel(
    r"$\mathrm{cm}^{-2}\,\mathrm{s}^{-1}\,\mathrm{sr}^{-1}\,\mathrm{MeV}^{-1}$"
)

ax1.set_yscale('symlog')

line1 = ax1.plot(
    np.arange(0, len(goes_16_proton_flux_mean), 1),
    goes_16_proton_flux_mean["P_tot"].to_numpy(),
    linewidth=2,
    label="Avg Proton Flux/h",
    zorder=3,
    color = "orange"
)

# -----------------------------------
# RIGHT AXIS (BACK)
# -----------------------------------

ax2 = ax1.twinx()

ax2.set_zorder(1)

ax2.set_ylabel("Count")

line2 = ax2.plot(
    np.arange(0, goes_16_seu_rate_df.shape[0], 1),
    goes_16_seu_rate_df["Count"],
    alpha=0.5,
    label="Total SEU/h",
    zorder=1
)

# -----------------------------------
# LEGEND
# -----------------------------------

lines = line1 + line2
labels = [l.get_label() for l in lines]

ax1.legend(lines, labels)

plt.show()

In [ ]:
X = proton_flux_seu_rate_df["Flux/h"].to_numpy().astype(np.float32).reshape(-1,1)
X = X[~np.isnan(X)].reshape(-1,1)
y = proton_flux_seu_rate_df["SEU/h"].to_numpy().astype(np.float32)
y = y[~np.isnan(y)]
X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X,y,test_size = .20)

In [ ]:
def Lin_Fit_SEU_Rate_Proton_Flux(a_X_train, a_X_test, a_y_train, a_y_test, a_model):
    a_X_test = np.log(a_X_test)
    a_X_train= np.log(a_X_train)

    a_model.fit(a_X_train, a_y_train)
    y_pred = a_model.predict(a_X_test)
    fig, ax = plt.subplots()
    ax.scatter(a_X_test, a_y_test, color = "b")
    ax.set_xlabel("Proton Flux")
    ax.set_title("SEU Rate vs Proton Flux " +type(a_model).__name__)
    ax.set_ylabel("SEU/hour")
    ax.plot(a_X_test, y_pred, color = 'k')
    plt.show()
    
    print("Mean Absolute Error:", sk.metrics.mean_absolute_error(y_true= y_test, y_pred = y_pred))
    print("Mean Sqaure Error:", sk.metrics.mean_squared_error(y_true = y_test, y_pred = y_pred))
    print("Root Mean Square Error:", np.sqrt(sk.metrics.mean_squared_error(y_true = y_test, y_pred = y_pred)))
    print("Model Score:", a_model.score(X_test,y_test))

Lin_Fit_SEU_Rate_Proton_Flux(X_train, X_test, y_train, y_test, sk.linear_model.LinearRegression())

The average linear regression model score tends to 0 indicating something wrong. non linear relationship between SEU occurences and proton fluxes? not enough occurences of SEUs in dataset? too noisy? Trying to compare SEU on GOES-16 with SOHO proton flux?

# EX 2. GOES 16 SEU Events with GOES Proton Flux
              GOES-13 Proton Fluence     GOES-13 Electron Fluence     Neutron
            --- Protons/cm2-day-sr ---  -- Electrons/cm2-day-sr --    Monitor

https://www.ncei.noaa.gov/data/goes-space-environment-monitor/access/avg/


In [ ]:
import os
PATH = "data/enviromental/goes/"
goes_proton_flux_files = os.listdir(PATH)

goes_proton_flux_df = pd.DataFrame()

for f in range(len(goes_proton_flux_files)):
    goes_file = pd.read_csv(PATH + goes_proton_flux_files[f],
                            skiprows = 716, sep = ",")
    goes_file["time_tag"] = pd.to_datetime(goes_file["time_tag"])
    #goes_file = goes_file.groupby(pd.Grouper(key = 'time_tag', freq = "h")).mean()
    goes_proton_flux_df = pd.concat([goes_proton_flux_df, goes_file]).reset_index(drop = True)
    goes_proton_flux_df = goes_proton_flux_df.sort_values(by = 'time_tag')

x = np.arange(0,goes_proton_flux_df.shape[0],1)


plt.bar(x,goes_proton_flux_df["ZPGT10E"])
plt.bar(x,goes_proton_flux_df["ZPGT10W"], alpha = .5)
plt.yscale("log")
plt.plot()